# Exploratory Data Analysis for Fashion Trends

**Purpose:** Demonstrate a reproducible EDA workflow for fashion trend tracking and visualization.

> **Data integrity note:** The bundled CSV is synthetic demonstration data created for this project. It is not a real market dataset and must not be presented as real consumer or retailer evidence. Replace it with a licensed public dataset before making real-world claims.

## Questions
1. How does search interest change over time?
2. Does the sales index appear to move with search interest in this example?
3. Which months have the highest sales index?
4. Is average price visibly associated with average rating?
5. What caveats should guide business decisions?

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_PATH = Path("../data/fashion_eda_sample.csv")
df = pd.read_csv(DATA_PATH)
df["month"] = pd.to_datetime(df["month"])

print("Shape:", df.shape)
display(df.head())
display(df.describe(include="all"))
print("\\nMissing values by column:")
display(df.isna().sum())
print("Exact duplicate rows:", df.duplicated().sum())

## 1. Data quality checks
Check types, missingness, duplicates, ranges, and category coverage before interpreting any plot.

In [ ]:
print(df.dtypes)
print("Date range:", df["month"].min(), "to", df["month"].max())
print("Categories:", sorted(df["category"].dropna().unique()))
assert df["search_interest_index"].between(0, 100).all()
assert df["avg_rating"].between(0, 5).all()
assert df["avg_price"].ge(0).all()

## 2. Trend line: search interest and sales index
A line chart helps reveal direction and possible seasonal turning points. The two measures use different units, so the second axis is used for display only.

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(df["month"], df["search_interest_index"], marker="o", label="Search interest index")
ax1.set_xlabel("Month")
ax1.set_ylabel("Relative search interest (0–100)")
ax1.tick_params(axis="x", rotation=45)

ax2 = ax1.twinx()
ax2.plot(df["month"], df["sales_index"], marker="s", label="Sales index")
ax2.set_ylabel("Illustrative sales index")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.title("Fashion Search Interest and Sales Index (Synthetic Example)")
plt.tight_layout()
plt.show()

## 3. Bar chart: category representation
This chart counts sample records per category. It does not measure actual market share or category sales.

In [ ]:
category_counts = df["category"].value_counts().sort_values(ascending=False)
category_counts.plot(kind="bar", figsize=(8, 4), title="Records per Category (Synthetic Example)")
plt.xlabel("Category")
plt.ylabel("Number of monthly records")
plt.tight_layout()
plt.show()

## 4. Pie chart: category mix
Pie charts are most suitable for a small number of mutually exclusive categories that sum to a whole. Here, the slices represent record counts only.

In [ ]:
plt.figure(figsize=(7, 5))
category_counts.plot(kind="pie", autopct="%1.1f%%", ylabel="", title="Category Share of Sample Records")
plt.tight_layout()
plt.show()

## 5. Scatter plot and correlation
Use scatter plots to inspect relationships. Correlation is not proof of causation and can be distorted by time trends or category mix.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["avg_price"], df["avg_rating"])
plt.xlabel("Average price (illustrative units)")
plt.ylabel("Average rating (0–5)")
plt.title("Average Price vs Rating (Synthetic Example)")
plt.tight_layout()
plt.show()

corr = df[["search_interest_index", "sales_index", "avg_price", "avg_rating", "review_count"]].corr(method="spearman")
display(corr.round(2))

## 6. Outlier checks

The IQR rule flags unusual values for investigation; it does not automatically mean they are errors.

In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return df.loc[(series < lower) | (series > upper), ["month", "category", series.name]]

for col in ["avg_price", "sales_index", "review_count"]:
    print(f"Potential outliers in {col}:")
    display(iqr_outliers(df[col]))

## 7. Example interpretation

Because this dataset is synthetic, these are demonstrations of how to write findings—not real market conclusions.

- Search interest and the sales index rise in the latter months in the constructed example.
- The highest values appear near the end of the year; real analysis should compare multiple years and account for promotions and holidays.
- Category record counts show the composition of this sample, not consumer market share.
- Price-rating relationships should be analysed by category and review volume before drawing conclusions.
- Validate any trend signal against an independent source and, where authorised, actual sell-through data.

## 8. Decision implications

For a real dataset, use sustained demand signals to propose small buying tests, align marketing calendars with verified audience interest, and review inventory availability before recommending category expansion. Track forecast error, sell-through, margin, stock-outs and return rate after any intervention.

## Limitations

This is a tutorial notebook with synthetic data. Replace the CSV with a publicly available dataset whose provenance and licence have been checked before reporting factual market findings.